<a href="https://colab.research.google.com/github/nmemranhussain/DiaVigil/blob/main/notebooks%20/%2001_diavigil_cohort_filtering_and_group_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

Mounted at /content/drive


**Cell 1: Environment & Directory Setup**

Mount Google Drive and create a structured directory layout to store raw data and processed parquet partitions.

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Establish project paths
BASE_DIR = '/content/drive/MyDrive/DiaVigil'
DATA_RAW = os.path.join(BASE_DIR, 'data/raw')
DATA_PROCESSED = os.path.join(BASE_DIR, 'data/processed')

os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)

print(f"Project directory ready at: {BASE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory ready at: /content/drive/MyDrive/DiaVigil


**Cell 2: Data Ingestion & Source Verification**

Download the official UCI Diabetes 130-US Hospitals dataset directly into the directory if not already stored, and load the raw encounter records.

In [3]:
import urllib.request
import zipfile
import pandas as pd

raw_csv_path = os.path.join(DATA_RAW, 'diabetic_data.csv')

if not os.path.exists(raw_csv_path):
    print("Fetching raw dataset from UCI Repository...")
    url = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
    zip_path = os.path.join(DATA_RAW, 'dataset.zip')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_RAW)
    print("Download and extraction complete.")
else:
    print("Raw dataset found locally.")

df_raw = pd.read_csv(raw_csv_path)

print(f"Raw Total Encounters: {len(df_raw):,}")
print(f"Unique Patient IDs: {df_raw['patient_nbr'].nunique():,}")

Fetching raw dataset from UCI Repository...
Download and extraction complete.
Raw Total Encounters: 101,766
Unique Patient IDs: 71,518


**Cell 3: Clinical Cohort Sanitization (CMS HRRP Criteria)**

Apply standard Centers for Medicare & Medicaid Services (CMS) exclusion rules: patients who expired during hospitalization or were discharged to hospice care cannot experience a 30-day readmission and must be excluded. Define the binary target variable.

In [4]:
# Exclusion codes from IDs_mapping.csv:
# 11: Expired
# 19: Expired at home
# 20: Expired in a medical facility
# 21: Expired, place unknown / hospice discharge
hospice_expired_dispositions = [11, 19, 20, 21]

cohort_df = df_raw[~df_raw['discharge_disposition_id'].isin(hospice_expired_dispositions)].copy()

# Target definition: 1 if readmitted in <30 days, 0 otherwise
cohort_df['target'] = (cohort_df['readmitted'] == '<30').astype(int)

excluded_count = len(df_raw) - len(cohort_df)
prevalence = cohort_df['target'].mean() * 100

print(f"Excluded Encounters (Hospice/Expired): {excluded_count:,}")
print(f"Retained Cohort Encounters: {len(cohort_df):,}")
print(f"Retained Unique Patients: {cohort_df['patient_nbr'].nunique():,}")
print(f"Cohort Readmission Rate (<30 days): {prevalence:.2f}%")

Excluded Encounters (Hospice/Expired): 1,652
Retained Cohort Encounters: 100,114
Retained Unique Patients: 70,439
Cohort Readmission Rate (<30 days): 11.34%


**Cell 4: Leak-Proof Stratified Group Partitioning**

Partition the dataset into an 80% Development Cohort (used for cross-validation, tuning, and calibration) and a 20% Untouched Audit Test Cohort (reserved strictly for final model evaluation). Ensure that all encounters for any given patient remain within a single partition.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

dev_idx, test_idx = next(
    sgkf.split(cohort_df, cohort_df['target'], groups=cohort_df['patient_nbr'])
)

dev_df = cohort_df.iloc[dev_idx].reset_index(drop=True)
audit_test_df = cohort_df.iloc[test_idx].reset_index(drop=True)

# Formal Leakage Check
shared_patients = set(dev_df['patient_nbr']).intersection(set(audit_test_df['patient_nbr']))
assert len(shared_patients) == 0, f"Critical Leakage: {len(shared_patients)} patients found in both sets!"

print("--- Partition Summary ---")
print(f"Dev Cohort:       {len(dev_df):,} encounters | Prevalence: {dev_df['target'].mean()*100:.2f}%")
print(f"Audit Test Cohort:{len(audit_test_df):,} encounters | Prevalence: {audit_test_df['target'].mean()*100:.2f}%")
print(f"Shared Patient IDs: {len(shared_patients)} (0 Patient-Level Identity Leakage)")

--- Partition Summary ---
Dev Cohort:       80,239 encounters | Prevalence: 11.36%
Audit Test Cohort:19,875 encounters | Prevalence: 11.27%
Shared Patient IDs: 0 (0 Patient-Level Identity Leakage)


**Cell 5: Export Sanitized Partitions to Parquet**

Serialize the partitions in Parquet format to preserve categorical encodings and data types without precision loss.

In [6]:
dev_path = os.path.join(DATA_PROCESSED, 'dev_cohort.parquet')
test_path = os.path.join(DATA_PROCESSED, 'audit_test_cohort.parquet')

dev_df.to_parquet(dev_path, index=False)
audit_test_df.to_parquet(test_path, index=False)

print(f"Saved Dev Cohort: {dev_path} ({os.path.getsize(dev_path) / 1024**2:.2f} MB)")
print(f"Saved Audit Test Cohort: {test_path} ({os.path.getsize(test_path) / 1024**2:.2f} MB)")

Saved Dev Cohort: /content/drive/MyDrive/DiaVigil/data/processed/dev_cohort.parquet (2.09 MB)
Saved Audit Test Cohort: /content/drive/MyDrive/DiaVigil/data/processed/audit_test_cohort.parquet (0.54 MB)
